# freetype_py smoke test

Smoke test for `freetype-py`, the Python bindings for the FreeType font engine.
Examples 2-5 open a small TrueType font that Example 2 builds in memory with
`fontTools` (`pip install fonttools`), so no font file has to be present on the
machine.

## Example 1: Library version and error handling

In [ ]:
import io

import freetype

# version() reports the FreeType library that freetype-py loaded.
version = freetype.version()
print("FreeType library version:", ".".join(map(str, version)))
assert len(version) == 3 and version[0] == 2, version

# Bytes that are not a font are rejected with FreeType's own error.
try:
    freetype.Face(io.BytesIO(b"\0" * 256))
    raise AssertionError("expected FT_Exception for data that is not a font")
except freetype.FT_Exception as e:
    print("not a font ->", e)

## Example 2: Open a TrueType font from memory

In [ ]:
import io

import freetype
from fontTools.fontBuilder import FontBuilder
from fontTools.pens.ttGlyphPen import TTGlyphPen


def outline(points):
    pen = TTGlyphPen(None)
    pen.moveTo(points[0])
    for point in points[1:]:
        pen.lineTo(point)
    pen.closePath()
    return pen.glyph()


# A four-glyph font: a triangle for "A", a rectangle for "B", and an empty space.
fb = FontBuilder(1000, isTTF=True)
fb.setupGlyphOrder([".notdef", "A", "B", "space"])
fb.setupCharacterMap({ord("A"): "A", ord("B"): "B", ord(" "): "space"})
fb.setupGlyf({
    ".notdef": outline([(50, 0), (50, 700), (450, 700), (450, 0)]),
    "A": outline([(50, 0), (300, 700), (550, 0)]),
    "B": outline([(80, 0), (80, 700), (540, 700), (540, 0)]),
    "space": TTGlyphPen(None).glyph(),
})
fb.setupHorizontalMetrics({".notdef": (500, 50), "A": (600, 50), "B": (620, 80), "space": (300, 0)})
fb.setupHorizontalHeader(ascent=800, descent=-200)
fb.setupNameTable({"familyName": "Demo", "styleName": "Regular"})
fb.setupOS2(sTypoAscender=800, usWinAscent=800, usWinDescent=200)
fb.setupPost()
buffer = io.BytesIO()
fb.save(buffer)
font_bytes = buffer.getvalue()

face = freetype.Face(io.BytesIO(font_bytes))
print(f"{face.family_name.decode()} {face.style_name.decode()}: {face.num_glyphs} glyphs, "
      f"{face.units_per_EM} units per em")
assert (face.family_name, face.style_name) == (b"Demo", b"Regular")
assert face.num_glyphs == 4 and face.units_per_EM == 1000
assert face.is_scalable

## Example 3: Render a glyph to a bitmap

In [ ]:
import io

import freetype

face = freetype.Face(io.BytesIO(font_bytes))
face.set_char_size(48 * 64)  # 48 pt at the default 72 dpi = 48 pixels per em
face.load_char("A")  # the default load flags render an anti-aliased bitmap
glyph = face.glyph
bitmap = glyph.bitmap
print(f"advance {glyph.advance.x / 64:.1f}px, bitmap {bitmap.width}x{bitmap.rows}")

# Advances are in 26.6 fixed point: 600 font units at 48 px per em is 28.8 px.
assert glyph.advance.x > 0
assert abs(glyph.advance.x / 64 - 600 * 48 / 1000) <= 1
assert bitmap.width > 0 and bitmap.rows > 0
assert len(bitmap.buffer) == bitmap.rows * bitmap.pitch
assert max(bitmap.buffer) > 0  # some pixels are inked

## Example 4: Character map and glyph names

In [ ]:
import io

import freetype

face = freetype.Face(io.BytesIO(font_bytes))
index_a = face.get_char_index("A")
index_b = face.get_char_index("B")
print("glyph index of A:", index_a, "B:", index_b)
assert (index_a, index_b) == (1, 2)
assert face.get_char_index("Z") == 0  # unmapped characters map to .notdef
assert face.get_glyph_name(index_a) == b"A"

mapped = {chr(code): index for code, index in face.get_chars() if index}
print("character map:", mapped)
assert mapped == {" ": 3, "A": 1, "B": 2}

## Example 5: Outlines in font units

In [ ]:
import io

import freetype

face = freetype.Face(io.BytesIO(font_bytes))
# FT_LOAD_NO_SCALE keeps the outline in font units, exactly as the font stores it.
face.load_char("A", freetype.FT_LOAD_NO_SCALE)
points = face.glyph.outline.points
print("A outline:", points, "contours end at", face.glyph.outline.contours)
assert points == [(50, 0), (300, 700), (550, 0)]
assert face.glyph.outline.contours == [2]  # one closed contour of three points
assert face.glyph.advance.x == 600

face.load_char("B", freetype.FT_LOAD_NO_SCALE)
assert len(face.glyph.outline.points) == 4
assert face.glyph.metrics.horiBearingX == 80
bbox = face.bbox
assert (bbox.xMin, bbox.yMin, bbox.xMax, bbox.yMax) == (50, 0, 550, 700)
print("font bounding box:", (bbox.xMin, bbox.yMin, bbox.xMax, bbox.yMax))